# 航空会社顧客満足度予測: LightGBM Baseline (Stratified 5-Fold CV)

- **コンペ名:** [Playground Series - Season 6, Episode 10](https://www.kaggle.com/competitions/playground-series-s6e10)
- **タスク:** 顧客アンケート等の表形式データに基づく満足度（2値: satisfied / not satisfied）の予測
- **評価指標:** **ROC AUC (Area Under the ROC Curve)**
- **モデル:** **LightGBM** (決定木勾配ブースティング)

---

## 1. 評価指標「ROC AUC」とは？
本コンペでは、単に正解/不正解を当てる正解率（Accuracy）ではなく、**「満足する確率（0.0〜1.0）」** を予測します。
ROC AUC は、予測確率の高い順に並べたとき、**本当に満足した乗客が上位に正しく並んでいるか（ランキングの正確さ）** を評価する指標です。
- `1.0`: 完璧（満点）
- `0.5`: 当てずっぽう（ランダム予測）
- `0.95以上`: 非常に優れた高精度モデル

## 2. ライブラリ読み込み & データ確認

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

# データのパスを自動探索（Kaggle環境・ローカル環境の両方に対応）
def find_data_dir():
    candidates = [
        '/kaggle/input/competitions/playground-series-s6e10',
        '/kaggle/input/playground-series-s6e10',
        '002_airline-satisfaction/data',
        '../data',
        'data'
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'train.csv')):
            return c
    matches = glob.glob('**/playground-series-s6e10/train.csv', recursive=True)
    if matches:
        return os.path.dirname(matches[0])
    raise FileNotFoundError('データファイルが見つかりません。')

data_dir = find_data_dir()
print(f'データ取得元: {data_dir}')

train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
sub_df = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))

print(f'訓練データ形状: {train_df.shape}')
print(f'テストデータ形状: {test_df.shape}')
train_df.head(3)

## 3. 前処理と特徴量の準備
- 目的変数 `satisfaction`（True/False）を 0 / 1 の整数に変換します。
- 文字列のカテゴリ変数（性別、旅行タイプ、座席クラスなど）を `category` 型に変換します。LightGBMはこれを自動で最適に分岐してくれます。

In [ ]:
target_col = 'satisfaction'
id_col = 'id'

y = train_df[target_col].astype(int).values
feature_cols = [c for c in train_df.columns if c not in [target_col, id_col]]

X = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
print(f'カテゴリ特徴量 ({len(cat_cols)}個): {cat_cols}')

for c in cat_cols:
    X[c] = X[c].astype('category')
    X_test[c] = X_test[c].astype('category')

print('前処理完了！')

## 4. 層化5分割交差検証（Stratified 5-Fold CV）による学習
- データを5つのグループに均等に分け、4つで学習し1つでテストすることを5回繰り返します（模試を5回受けるイメージ）。
- 全体を通じた手元検証スコア（OOF ROC AUC）を算出し、本番スコアの確実な指標とします。

In [ ]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))
models = []
fold_scores = []

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'n_estimators': 500,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, y_tr = X.iloc[train_idx], y[train_idx]
    X_va, y_val = X.iloc[val_idx], y[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    
    val_pred = model.predict_proba(X_va)[:, 1]
    oof_preds[val_idx] = val_pred
    
    score = roc_auc_score(y_val, val_pred)
    fold_scores.append(score)
    models.append(model)
    print(f'Fold {fold}/{n_splits} - ROC AUC: {score:.5f}')
    
    test_preds += model.predict_proba(X_test)[:, 1] / n_splits

overall_auc = roc_auc_score(y, oof_preds)
print('=' * 40)
print(f'全体 OOF ROC AUC (手元検証スコア): {overall_auc:.5f}')
print('=' * 40)

## 5. 特徴量重要度（Feature Importance）の可視化
LightGBM が「乗客の満足度を予測する上で、どの項目を最も重視したか」をグラフで確認します。

In [ ]:
importances = np.mean([m.feature_importances_ for m in models], axis=0)
fi_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=fi_df, x='importance', y='feature', palette='viridis')
plt.title('LightGBM Feature Importance (Top Features for Satisfaction)')
plt.xlabel('Importance (Split Count)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 6. 提出ファイルの出力と検証
5つのモデルによる予測確率の平均値を `submission.csv` に出力します。

In [ ]:
sub_df['satisfaction'] = test_preds
sub_df.to_csv('submission.csv', index=False)

print('提出ファイル作成完了: submission.csv')
print(f'- 行数: {len(sub_df)} (期待値: 299844)')
print(f'- 欠損値数: {sub_df["satisfaction"].isnull().sum()}')
print(f'- 予測確率の範囲: {sub_df["satisfaction"].min():.4f} 〜 {sub_df["satisfaction"].max():.4f}')
print('\n先頭5行のプレビュー:')
print(sub_df.head())